# ITSN1 Carriers Investigation

- Author: Sheila Yeboah
- Date Modified: 3/11/26
- Description: Investigating carriers of Bonferonni correction passing ITSN1 variants in the UK Biobank Cohort

## Imports

In [2]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import os
import os.path
import subprocess
import pyspark
import dxpy
import dxdata

## Initialize Spark

In [3]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

## Set Variants of Interest

In [ ]:
# make variants of interest list, nominated by Bonferonni in UK Biobank
GENE = "ITSN1"
data = "WGS"
# dict maps ancestry to nominated variants within that cohort
voi_dict = {"EUR":["chr21:33772205:G:A"]}

# dict maps nominated variant to its rsid for filenames
rsid_dict = {"chr21:33772205:G:A":"rs779207769"}

voi_dict

{'EUR': ['chr21:33772205:G:A']}

## Pull .raw files to local environments

In [5]:
# download the necessary .raw files

for ancestry in voi_dict.keys():
    OUTPUT_DIR =  "/results/11625_ITSN1_WGS_Pan_Ancestry"
    subprocess.run(
        ["dx", "download", f"{OUTPUT_DIR}/{ancestry}/{ancestry}_{GENE}_{data}.raw"],
        check=True
    )

## Extract Variant Carriers

In [ ]:
# function to go into folder, and pull the carrier id
def grab_variant_carriers(voi_dict,rsid_dict, data):
    """
    Function takes in dictionary where keys are ancestry groups, and values are list of variants
    that passed Bonferroni correction.
    Params:
    voi_dict: Dictionary mapping ancestry groups to nominated variants
    rsid_dict: Dictionary mapping variant to rsids
    data: str, whether data is imputed array (imp) or whole genomes (wgs)

    Returns: tuple, carriers_df (Dataframe), and carriers_dict (Dictionary)
    """
    OUTPUT_DIR =  "/results/11625_ITSN1_WGS_Pan_Ancestry"
    for ancestry in voi_dict.keys():
        
        # loop through variants in list for ancestry
        for variant in voi_dict[ancestry]:
            # read in .raw file to get variants of interest
    
            raw = pd.read_csv(f"{ancestry}_{GENE}_{data.upper()}.raw", sep="\s+")
    
            # keep only cases 
            cases_data = raw[raw["PHENOTYPE"] == 2]
    
            # get variant column name
            variant_id = f"{variant}_{(variant.rsplit(':', 1)[-1])}"
            print(variant_id)

            carriers_df = cases_data[cases_data[variant_id] > 0].copy()
            carriers_df.rename(columns={variant_id:variant}, inplace=True)
            carriers_df = carriers_df[["IID", variant]]
            carriers_list = carriers_df["IID"].tolist()
            rsid = rsid_dict[variant]
            
            #add ancestry column
            carriers_df["Ancestry"] = ancestry
            
            carriers_df.to_csv(f"{ancestry}_{rsid}_carriers.csv", index=False)
            
            # upload to dnanexus
            subprocess.run(["dx", "upload", f"{ancestry}_{rsid}_carriers.csv", "--path", f"{OUTPUT_DIR}/ITSN1_carriers/"],check=True)
            
            carrier_dict = {variant:carriers_list}
            print(carriers_df)
            
            return carriers_df, carrier_dict

## Create clinical information dataset

In [7]:
# pull participant information using Spark
dispensed_dataset_id = dxpy.find_one_data_object(typename="Dataset", name="app*.dataset", folder="/", name_mode="glob")["id"]
dataset = dxdata.load_dataset(id=dispensed_dataset_id)
participant = dataset["participant"]

In [ ]:
# pull demographic and clinical information (ids got from table exporter)


# Filter by appropriate fields
field_names = [
    # ---- demographic information ----
    "eid", # IID
    "p31", # GENETIC_SEX
    "p34", # BIRTH_YEAR
    "p52", # MONTH OF BIRTH
    "p21022", # AGE_OF_RECRUIT
    "p40000_i0", # DATE_OF_DEATH
    "p1647_i0", # Country of Birth (UK/elsewhere)
    "p20115_i0", # "Country of birth (non UK origin)"
    "p22189", # Townsend deprivation index at recruitment
    "p21000_i0", # Ethnic_Background
    "p26414", #Education score (England)
    "p26431", #Education score (Scotland)
    "p26421", # Education score (Wales)
    
    
    # ---- Family History -----
    "p20107_i0", # ILLNESS_OF_FATHER_i0
    "p20107_i1", # ILLNESS_OF_FATHER_i1
    "p20110_i0", # ILLNESS_OF_MOTHER_i0
    "p20110_i1", # ILLNESS_OF_MOTHER_i1
    "p20111_i0", # ILLNESS_OF_SIBLINGS_i0
    "p20111_i1", # ILLNESS_OF_SIBLINGS_i1
    
   
    # --- Individual Health History ---
    "p42032", # "Date of Parkinson's disease report" 
    "p41270", # "Diagnoses, ICD-10"
    "p41202", # "Diagnoses, main ICD-10"
    "p41271", # "Diagnoses, ICD-9"
    "p41203", #  "Diagnoses, main ICD-9"
    "p41204", #"Diagnoses, secondary ICD-10"
    "p41205",#"Diagnoses, secondary ICD-9"
    
    # ---- Smell -----
    "p28612", #Currently suffering from a loss or change in sense of smell
    "p28613", #Length of time suffering from a loss or change in smell
    "p28614", #Extent affected by a loss or change in sense of smell
    
    # --- Vitals ----
    "p50_i0", # Standing Height (cm) _i0
    "p4079_i0_a0", # Diastolic bp automatic_i0
    "p4080_i0_a0", # Systolic bp automatic_i0
    "p21001_i0", # BMI_i0 (kg/m^2)
    "p21002_i0", # weight (kg)
    
    # ---Lifestyle/Environmental----
    "p20160_i0", # Ever smoked_i0
    "p20116_i0", # Smoking status_i0
    "p2090_i0", #Seen doctor (GP) for nerves, anxiety, tension or depression
    "p29000", #Mental health conditions ever diagnosed by a professional
    "p29001" #Mental health conditions experienced by first degree relative
     
    
]

full_df = participant.retrieve_fields(engine=dxdata.connect(dialect="hive+pyspark"),names=field_names, coding_values="replace")

# remove columns with all NA
full_df = full_df.toPandas().dropna(axis=1, how="all")

In [ ]:
# get carriers df
carriers_df = grab_variant_carriers(voi_dict,rsid_dict, "wgs")[0]

In [12]:
full_df.rename(
    columns = {

    # ---- Demographic Information ----
    "eid": "IID",
    "p31": "Genetic_Sex",
    "p34": "Birth_Year",
    "p52": "Birth_Month",
    "p21022": "Age_At_Recruitment",
    "p40000_i0": "Date_of_Death",
    "p1647_i0": "Country_of_Birth_UK_or_Elsewhere",
    "p20115_i0": "Country_of_Birth_Non_UK",
    "p22189": "Townsend_Deprivation_Index_At_Recruitment",
    "p21000_i0": "Ethnic_Background",
    "p26414": "Education_Score_England",
    "p26431": "Education_Score_Scotland",
    "p26421": "Education_Score_Wales",

    # ---- Family History ----
    "p20107_i0": "Illness_of_Father_Instance0",
    "p20107_i1": "Illness_of_Father_Instance1",
    "p20110_i0": "Illness_of_Mother_Instance0",
    "p20110_i1": "Illness_of_Mother_Instance1",
    "p20111_i0": "Illness_of_Siblings_Instance0",
    "p20111_i1": "Illness_of_Siblings_Instance1",

    # ---- Individual Health History ----
    "p42032": "Parkinsons_Disease_Report_Date",
    "p41270": "Diagnoses_ICD10_All",
    "p41202": "Diagnoses_ICD10_Main",
    "p41271": "Diagnoses_ICD9_All",
    "p41203": "Diagnoses_ICD9_Main",
    "p41204": "Diagnoses_ICD10_Secondary",
    "p41205": "Diagnoses_ICD9_Secondary",

    # ---- Smell ----
    "p28612": "Current_Loss_or_Change_in_Smell",
    "p28613": "Duration_of_Smell_Loss_or_Change",
    "p28614": "Extent_of_Smell_Loss_or_Change",

    # ---- Vitals ----
    "p50_i0": "Standing_Height_cm_Instance0",
    "p4079_i0_a0": "Diastolic_BP_Automatic_Instance0",
    "p4080_i0_a0": "Systolic_BP_Automatic_Instance0",
    "p21001_i0": "BMI_kg_per_m2_Instance0",
    "p21002_i0": "Weight_kg_Instance0",

    # ---- Lifestyle / Environmental ----
    "p20160_i0": "Ever_Smoked_Instance0",
    "p20116_i0": "Smoking_Status_Instance0",
    "p2090_i0": "Seen_GP_for_Anxiety_or_Depression_Instance0",
    "p29000": "Mental_Health_Conditions_Diagnosed",
    "p29001": "Mental_Health_Conditions_in_First_Degree_Relative"
}, inplace=True)

In [ ]:
# save dataframe as clinical data and upload 
full_df.to_csv("clinical_data_for_ITSN1_carriers.csv", index=False)

OUTPUT_DIR =  "/results/11625_ITSN1_WGS_Pan_Ancestry/ITSN1_carriers"

subprocess.run(["dx", "upload", f"clinical_data_for_ITSN1_carriers.csv", "--path", f"{OUTPUT_DIR}/"],check=True)

In [ ]:
# merge carrier df with clinical data

carriers_df["IID"] = carriers_df["IID"].astype(str)
full_df["IID"] = full_df["IID"].astype(str)
for ancestry in voi_dict.keys():
    for variant in voi_dict[ancestry]:
        rsid = rsid_dict[variant]
        merged_df = carriers_df.merge(full_df, on="IID", how="left")
        merged_df_clean = merged_df.copy()

        # add variant as SNP column
        merged_df_clean.insert(1, "SNP", merged_df.columns[1])
        # add column for carrier_status
        print(type(variant))
        print(type(merged_df[variant]))
        merged_df_clean.insert(2, "carrier_status", merged_df[variant].astype(int).map({1: "heterozygous", 2: "homozygous"}))
        merged_df_clean.drop(columns=[variant], inplace=True)

        # also drop any columns that are all NAs

        merged_df_clean.dropna(axis=1, how="all", inplace=True)
        merged_df_clean.to_csv(f"{ancestry}_{rsid}_clindata.csv", index=False)